# Temp Notebook - Clear KC Tags on 100% Problems (Student 14475)

This notebook:
1. Uses helper functions from `utils/dataset.py` to compute best attempts.
2. Finds problems where student `14475` has score `1.0` (100%).
3. Clears only `annotations[problem_id]["gaps"]` for those problems.
4. Keeps the annotation JSON structure and all other fields untouched.

In [ ]:
from pathlib import Path
import sys
import json
from copy import deepcopy

# Ensure project root is importable when running from experiments/evaluations
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'utils').exists() is False:
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.dataset import load_joined_datasets, get_best_attempts

STUDENT_ID = 14475
INPUT_ANNOTATION_PATH = PROJECT_ROOT / 'dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14475_1775593132134.json'
OUTPUT_ANNOTATION_PATH = PROJECT_ROOT / 'dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14475_1775593132134_temp_no_kc_on_100.json'

print('Project root:', PROJECT_ROOT)
print('Input annotation:', INPUT_ANNOTATION_PATH)
print('Output annotation:', OUTPUT_ANNOTATION_PATH)

In [ ]:
# Use shared helper functions from utils/dataset.py
full_data = load_joined_datasets(verbose=False)
best_attempts = get_best_attempts(full_data, verbose=False)

student_best = best_attempts[best_attempts['SubjectID'] == STUDENT_ID].copy()
perfect_problem_ids = set(
    student_best.loc[student_best['Score'] >= 1.0, 'ProblemID']
    .astype(int)
    .astype(str)
)

print(f'Best-attempt rows for {STUDENT_ID}: {len(student_best)}')
print(f'Problems with 100% score: {len(perfect_problem_ids)}')
print('Problem IDs (100%):', sorted(perfect_problem_ids, key=lambda x: int(x))[:30], '...')

In [ ]:
with open(INPUT_ANNOTATION_PATH, 'r', encoding='utf-8') as f:
    original_data = json.load(f)

updated_data = deepcopy(original_data)
annotations = updated_data.get('annotations', {})

# Only modify the gaps list for perfect-score problems that exist in annotation file
changed_problem_ids = []
for pid in sorted(perfect_problem_ids, key=lambda x: int(x)):
    if pid in annotations and isinstance(annotations[pid], dict) and 'gaps' in annotations[pid]:
        if annotations[pid]['gaps']:
            changed_problem_ids.append(pid)
        annotations[pid]['gaps'] = []

print(f'Annotated problems in file: {len(annotations)}')
print(f'Perfect-score problems found in annotation file: {sum(1 for p in perfect_problem_ids if p in annotations)}')
print(f'Problems whose non-empty gaps were cleared: {len(changed_problem_ids)}')
print('Changed problem IDs:', changed_problem_ids)

In [ ]:
with open(OUTPUT_ANNOTATION_PATH, 'w', encoding='utf-8') as f:
    json.dump(updated_data, f, indent=2, ensure_ascii=False)

print('Saved patched annotation file to:')
print(OUTPUT_ANNOTATION_PATH)

In [ ]:
# Validation: structure and untouched data checks
top_level_same = set(original_data.keys()) == set(updated_data.keys())
annotation_keys_same = set(original_data.get('annotations', {}).keys()) == set(updated_data.get('annotations', {}).keys())

print('Top-level keys unchanged:', top_level_same)
print('Problem keys unchanged:', annotation_keys_same)

# For non-target problems, full annotation entry should be identical
non_target_equal = True
for pid, original_entry in original_data.get('annotations', {}).items():
    if pid not in perfect_problem_ids:
        if original_entry != updated_data['annotations'].get(pid):
            non_target_equal = False
            print('Mismatch on non-target problem:', pid)
            break

print('Non-target problems untouched:', non_target_equal)